In [1]:
import os
import numpy as np
import pandas as pd

from nltk.corpus import stopwords

from sklearn.base import clone
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score

## Data

In [2]:
outputfiles = ["caes_feats", "hablacultura_feats", "kwiziq_feats"]

dfs = []

for file in outputfiles:
    df = pd.read_csv(os.path.join("data", file + ".csv"))
    
    # add dataset column (remove "_feats")
    df["dataset"] = file.replace("_feats", "")
    
    dfs.append(df)

# concatenate all
data = pd.concat(dfs, ignore_index=True)
data.groupby("dataset")["cefr"].value_counts()

dataset       cefr
caes          A2      5708
              A1      5553
              B1      4489
              B2      3100
              C1      1786
hablacultura  B2       273
              B1       245
              C1        99
              A2        33
kwiziq        B2        59
              B1        54
              C1        39
              A2        33
              A1        21
Name: count, dtype: int64

## Model Training 5-fold CV

Feature group combinations

In [3]:
import numpy as np
import pandas as pd

from nltk.corpus import stopwords

from sklearn.base import clone
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score, accuracy_score, f1_score
from sklearn.metrics import confusion_matrix


# =========================================================
# SETTINGS
# =========================================================
target = "cefr"
text_col = "lemmas"

spanish_stopwords = stopwords.words("spanish")

textstat_features = [
    "fernandez_huerta", "szigriszt_pazos", "gutierrez_polini", "crawford",
    "syllable_count", "lexicon_count", "sentence_count", "words_per_sentence",
    "avg_syllables_per_word", "difficult_words"
]

overlap_features = ["A1_overlap", "A2_overlap", "B1_overlap", "B2_overlap"]

# CEFR mapping for QWK
cefr_order = ["A1", "A2", "B1", "B2", "C1"]
cefr_to_int = {c: i for i, c in enumerate(cefr_order)}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


# =========================================================
# FINAL MODEL
# =========================================================
def make_model(feature_pipeline):
    return Pipeline([
        ("features", feature_pipeline),
        ("scaler", StandardScaler(with_mean=False)),
        ("clf", LogisticRegression(
            solver="lbfgs",
            penalty="l2",
            C=1.0,
            max_iter=3000,
            class_weight="balanced"
        ))
    ])


# =========================================================
# FEATURE PIPELINES
# =========================================================
def numeric_pipe(cols):
    return Pipeline([
        ("select", FunctionTransformer(lambda X: X[cols], validate=False))
    ])


def text_pipe():
    return Pipeline([
        ("select", FunctionTransformer(lambda X: X[text_col], validate=False)),
        ("tfidf", TfidfVectorizer(
            stop_words=spanish_stopwords,
            ngram_range=(1, 1)
        ))
    ])


# =========================================================
# FEATURE COMBINATIONS
# =========================================================
feature_configs = {
    "textstat": numeric_pipe(textstat_features),
    "overlap": numeric_pipe(overlap_features),
    "ngrams": text_pipe(),

    "textstat+overlap": FeatureUnion([
        ("textstat", numeric_pipe(textstat_features)),
        ("overlap", numeric_pipe(overlap_features)),
    ]),

    "textstat+ngrams": FeatureUnion([
        ("textstat", numeric_pipe(textstat_features)),
        ("ngrams", text_pipe()),
    ]),

    "overlap+ngrams": FeatureUnion([
        ("overlap", numeric_pipe(overlap_features)),
        ("ngrams", text_pipe()),
    ]),

    "all": FeatureUnion([
        ("textstat", numeric_pipe(textstat_features)),
        ("overlap", numeric_pipe(overlap_features)),
        ("ngrams", text_pipe()),
    ]),
}


# =========================================================
# CV LOOP
# =========================================================
rows = []
cm_md_sections = []

labels = list(range(len(cefr_order)))

for ds_name, group in data.groupby("dataset"):
    print(f"\n=== DATASET: {ds_name} ===")

    X_df = group.copy()

    y_str = group[target].astype(str).str.upper().values
    y = np.array([cefr_to_int[v] for v in y_str])

    for feat_name, feat_pipe in feature_configs.items():
        print(f"\n--- Features: {feat_name} ---")

        model = make_model(feat_pipe)

        qwk_list, acc_list, f1w_list = [], [], []
        all_y_true = []
        all_y_pred = []

        for fold_id, (tr_idx, te_idx) in enumerate(cv.split(X_df, y), start=1):
            X_tr, X_te = X_df.iloc[tr_idx], X_df.iloc[te_idx]
            y_tr, y_te = y[tr_idx], y[te_idx]

            clf = clone(model)
            clf.fit(X_tr, y_tr)
            y_pred = clf.predict(X_te)

            all_y_true.extend(y_te)
            all_y_pred.extend(y_pred)

            qwk = cohen_kappa_score(y_te, y_pred, weights="quadratic")
            acc = accuracy_score(y_te, y_pred)
            f1w = f1_score(y_te, y_pred, average="weighted")

            qwk_list.append(qwk)
            acc_list.append(acc)
            f1w_list.append(f1w)

            print(
                f"Fold {fold_id}: QWK={qwk:.4f}, acc={acc:.4f}, f1_weighted={f1w:.4f}"
            )

        cm = confusion_matrix(all_y_true, all_y_pred, labels=labels)

        cm_df = pd.DataFrame(
            cm,
            index=[f"True {c}" for c in cefr_order],
            columns=[f"Pred {c}" for c in cefr_order]
        )

        cm_md_sections.append(f"## Dataset: {ds_name} | Features: {feat_name}\n")
        cm_md_sections.append(cm_df.to_markdown())
        cm_md_sections.append("\n")

        row = {
            "dataset": ds_name,
            "features": feat_name,
            "qwk_mean": np.mean(qwk_list),
            "qwk_std": np.std(qwk_list),
            "acc_mean": np.mean(acc_list),
            "acc_std": np.std(acc_list),
            "f1_weighted_mean": np.mean(f1w_list),
            "f1_weighted_std": np.std(f1w_list),
        }

        rows.append(row)

        print(
            f"Mean: QWK={row['qwk_mean']:.4f} ± {row['qwk_std']:.4f} | "
            f"acc={row['acc_mean']:.4f} ± {row['acc_std']:.4f} | "
            f"f1_weighted={row['f1_weighted_mean']:.4f} ± {row['f1_weighted_std']:.4f}"
        )


# =========================================================
# RESULTS
# =========================================================
results = pd.DataFrame(rows).sort_values(
    by=["dataset", "f1_weighted_mean"],
    ascending=[True, False]
)

print("\n=== FEATURE COMBINATION RESULTS ===")
print(results)

best_per_dataset = results.groupby("dataset", as_index=False).first()

print("\n=== BEST FEATURE SET PER DATASET ===")
print(best_per_dataset)

results.to_csv("exp1_features.csv", index=False)

with open("confusion_matrices.md", "w", encoding="utf-8") as f:
    f.write("# Confusion Matrices\n\n")
    f.write("\n".join(cm_md_sections))


=== DATASET: caes ===

--- Features: textstat ---
Fold 1: QWK=0.6866, acc=0.5363, f1_weighted=0.5319
Fold 2: QWK=0.6968, acc=0.5406, f1_weighted=0.5354
Fold 3: QWK=0.7003, acc=0.5328, f1_weighted=0.5272
Fold 4: QWK=0.7093, acc=0.5452, f1_weighted=0.5386
Fold 5: QWK=0.6947, acc=0.5411, f1_weighted=0.5362
Mean: QWK=0.6975 ± 0.0074 | acc=0.5392 ± 0.0042 | f1_weighted=0.5339 ± 0.0040

--- Features: overlap ---
Fold 1: QWK=0.6137, acc=0.4671, f1_weighted=0.4624
Fold 2: QWK=0.6453, acc=0.4795, f1_weighted=0.4771
Fold 3: QWK=0.6482, acc=0.4718, f1_weighted=0.4659
Fold 4: QWK=0.6345, acc=0.4747, f1_weighted=0.4708
Fold 5: QWK=0.6318, acc=0.4677, f1_weighted=0.4625
Mean: QWK=0.6347 ± 0.0122 | acc=0.4721 ± 0.0046 | f1_weighted=0.4678 ± 0.0056

--- Features: ngrams ---
Fold 1: QWK=0.9838, acc=0.9872, f1_weighted=0.9872
Fold 2: QWK=0.9889, acc=0.9891, f1_weighted=0.9891
Fold 3: QWK=0.9857, acc=0.9884, f1_weighted=0.9884
Fold 4: QWK=0.9905, acc=0.9910, f1_weighted=0.9910
Fold 5: QWK=0.9860, acc=0.

## Cross-Dataset

In [4]:
import numpy as np
import pandas as pd

from nltk.corpus import stopwords

from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score


# =========================================================
# SETTINGS
# =========================================================
target = "cefr"
text_col = "lemmas"   # or "text"

textstat_features = [
    "fernandez_huerta", "szigriszt_pazos", "gutierrez_polini", "crawford",
    "syllable_count", "lexicon_count", "sentence_count", "words_per_sentence",
    "avg_syllables_per_word", "difficult_words"
]

overlap_features = ["A1_overlap", "A2_overlap", "B1_overlap", "B2_overlap"]

spanish_stopwords = stopwords.words("spanish")

cefr_order = ["A1", "A2", "B1", "B2", "C1"]
cefr_to_int = {label: idx for idx, label in enumerate(cefr_order)}
int_to_cefr = {idx: label for label, idx in cefr_to_int.items()}

# Final selected model settings
LOGREG_KWARGS = dict(
    solver="lbfgs",
    penalty="l2",
    C=1.0,
    max_iter=3000,
    class_weight="balanced"
)

# Feature-group combinations to test
feature_group_configs = {
    "textstat": ["textstat"],
    "overlap": ["overlap"],
    "ngrams": ["ngrams"],
    "textstat+overlap": ["textstat", "overlap"],
    "textstat+ngrams": ["textstat", "ngrams"],
    "overlap+ngrams": ["overlap", "ngrams"],
    "textstat+overlap+ngrams": ["textstat", "overlap", "ngrams"],
}


# =========================================================
# HELPERS
# =========================================================
def build_feature_matrices(train_df, test_df, groups):
    X_train_parts = []
    X_test_parts = []

    if "textstat" in groups:
        scaler_textstat = StandardScaler()
        Xtr_textstat = scaler_textstat.fit_transform(
            train_df[textstat_features].astype(float).values
        )
        Xte_textstat = scaler_textstat.transform(
            test_df[textstat_features].astype(float).values
        )
        X_train_parts.append(csr_matrix(Xtr_textstat))
        X_test_parts.append(csr_matrix(Xte_textstat))

    if "overlap" in groups:
        scaler_overlap = StandardScaler()
        Xtr_overlap = scaler_overlap.fit_transform(
            train_df[overlap_features].astype(float).values
        )
        Xte_overlap = scaler_overlap.transform(
            test_df[overlap_features].astype(float).values
        )
        X_train_parts.append(csr_matrix(Xtr_overlap))
        X_test_parts.append(csr_matrix(Xte_overlap))

    if "ngrams" in groups:
        tfidf = TfidfVectorizer(
            stop_words=spanish_stopwords,
            ngram_range=(1, 1)
        )
        Xtr_text = train_df[text_col].fillna("").astype(str).values
        Xte_text = test_df[text_col].fillna("").astype(str).values

        Xtr_ngrams = tfidf.fit_transform(Xtr_text)
        Xte_ngrams = tfidf.transform(Xte_text)

        X_train_parts.append(Xtr_ngrams)
        X_test_parts.append(Xte_ngrams)

    if len(X_train_parts) == 1:
        X_train = X_train_parts[0]
        X_test = X_test_parts[0]
    else:
        X_train = hstack(X_train_parts).tocsr()
        X_test = hstack(X_test_parts).tocsr()

    return X_train, X_test


def evaluate_transfer(train_df, test_df, groups):
    X_train, X_test = build_feature_matrices(train_df, test_df, groups)

    y_train = train_df[target].astype(str).str.upper().map(cefr_to_int).values
    y_test = test_df[target].astype(str).str.upper().map(cefr_to_int).values

    clf = LogisticRegression(**LOGREG_KWARGS)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    qwk = cohen_kappa_score(y_test, y_pred, weights="quadratic")
    acc = accuracy_score(y_test, y_pred)
    f1w = f1_score(y_test, y_pred, average="weighted", zero_division=0)
    f1m = f1_score(y_test, y_pred, average="macro", zero_division=0)

    per_class = f1_score(
        y_test,
        y_pred,
        average=None,
        labels=np.arange(len(cefr_order)),
        zero_division=0
    )

    row = {
        "train": "caes",
        "test": test_df["dataset"].iloc[0],
        "features": "+".join(groups),
        "qwk": float(qwk),
        "accuracy": float(acc),
        "f1_weighted": float(f1w),
        "f1_macro": float(f1m),
    }

    for ci, cname in enumerate(cefr_order):
        row[f"f1_{cname}"] = float(per_class[ci])

    return row


# =========================================================
# DATA SPLIT
# =========================================================
train_df = data[data["dataset"] == "caes"].copy()
test_datasets = ["hablacultura", "kwiziq"]

print("Train set: caes")
print(train_df[target].astype(str).str.upper().value_counts().reindex(cefr_order, fill_value=0))

for ds_name in test_datasets:
    test_df = data[data["dataset"] == ds_name].copy()
    print(f"\nTest set: {ds_name}")
    print(test_df[target].astype(str).str.upper().value_counts().reindex(cefr_order, fill_value=0))


# =========================================================
# CROSS-DATASET FEATURE SEARCH
# Train on CAES, test on the others
# =========================================================
rows = []

for test_name in test_datasets:
    test_df = data[data["dataset"] == test_name].copy()

    print(f"\n{'=' * 80}")
    print(f"TRAIN: caes | TEST: {test_name}")
    print(f"{'=' * 80}")

    for feat_name, groups in feature_group_configs.items():
        row = evaluate_transfer(train_df, test_df, groups)
        row["feature_config_name"] = feat_name
        rows.append(row)

        print(
            f"{feat_name:24s} | "
            f"QWK={row['qwk']:.4f} | "
            f"acc={row['accuracy']:.4f} | "
            f"f1_weighted={row['f1_weighted']:.4f} | "
            f"f1_macro={row['f1_macro']:.4f}"
        )


# =========================================================
# RESULTS
# =========================================================
results = pd.DataFrame(rows)

results = results.sort_values(
    by=["test", "f1_weighted", "qwk", "accuracy"],
    ascending=[True, False, False, False]
)

print("\n=== CROSS-DATASET RESULTS ===")
print(results)

print("\n=== BEST FEATURE CONFIG PER TEST DATASET ===")
best_per_test = results.groupby("test", as_index=False).first()
print(
    best_per_test[
        ["train", "test", "feature_config_name", "features", "qwk", "accuracy", "f1_weighted", "f1_macro"]
    ]
)

print("\n=== LATEX TABLE ===")
latex_df = results[
    ["train", "test", "feature_config_name", "qwk", "accuracy", "f1_weighted", "f1_macro"]
].copy()
print(latex_df.to_latex(index=False, float_format="%.4f"))

# Optional saves
results.to_csv("exp2_crossdataset.csv", index=False)

Train set: caes
cefr
A1    5553
A2    5708
B1    4489
B2    3100
C1    1786
Name: count, dtype: int64

Test set: hablacultura
cefr
A1      0
A2     33
B1    245
B2    273
C1     99
Name: count, dtype: int64

Test set: kwiziq
cefr
A1    21
A2    33
B1    54
B2    59
C1    39
Name: count, dtype: int64

TRAIN: caes | TEST: hablacultura
textstat                 | QWK=0.1069 | acc=0.2323 | f1_weighted=0.2690 | f1_macro=0.1612
overlap                  | QWK=0.0110 | acc=0.0754 | f1_weighted=0.1039 | f1_macro=0.0787
ngrams                   | QWK=0.1409 | acc=0.2615 | f1_weighted=0.2441 | f1_macro=0.1875
textstat+overlap         | QWK=0.1104 | acc=0.2462 | f1_weighted=0.2812 | f1_macro=0.1713
textstat+ngrams          | QWK=0.1572 | acc=0.3185 | f1_weighted=0.3383 | f1_macro=0.2381
overlap+ngrams           | QWK=0.0748 | acc=0.2185 | f1_weighted=0.2188 | f1_macro=0.1678
textstat+overlap+ngrams  | QWK=0.1563 | acc=0.3308 | f1_weighted=0.3496 | f1_macro=0.2483

TRAIN: caes | TEST: kwiziq
textsta

## Training Size

In [5]:
import os
import numpy as np
import pandas as pd

from nltk.corpus import stopwords

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score


# =========================================================
# LOAD DATA
# =========================================================
outputfiles = ["caes_feats", "hablacultura_feats", "kwiziq_feats"]

dfs = []
for file in outputfiles:
    df = pd.read_csv(os.path.join("data", file + ".csv"))
    df["dataset"] = file.replace("_feats", "")
    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)

print(data.groupby("dataset")["cefr"].value_counts())


# =========================================================
# SETTINGS
# =========================================================
target = "cefr"
text_col = "lemmas"
random_state = 42

spanish_stopwords = stopwords.words("spanish")

textstat_features = [
    "fernandez_huerta", "szigriszt_pazos", "gutierrez_polini", "crawford",
    "syllable_count", "lexicon_count", "sentence_count", "words_per_sentence",
    "avg_syllables_per_word", "difficult_words"
]

overlap_features = ["A1_overlap", "A2_overlap", "B1_overlap", "B2_overlap"]

cefr_order = ["A1", "A2", "B1", "B2", "C1"]
cefr_to_int = {c: i for i, c in enumerate(cefr_order)}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)


# =========================================================
# FINAL MODEL
# =========================================================
def make_model(feature_pipeline):
    return Pipeline([
        ("features", feature_pipeline),
        ("scaler", StandardScaler(with_mean=False)),
        ("clf", LogisticRegression(
            solver="lbfgs",
            penalty="l2",
            C=1.0,
            max_iter=3000,
            class_weight="balanced"
        ))
    ])


# =========================================================
# FEATURE PIPELINES
# =========================================================
def numeric_pipe(cols):
    return Pipeline([
        ("select", FunctionTransformer(lambda X: X[cols], validate=False))
    ])


def text_pipe():
    return Pipeline([
        ("select", FunctionTransformer(lambda X: X[text_col], validate=False)),
        ("tfidf", TfidfVectorizer(
            stop_words=spanish_stopwords,
            ngram_range=(1, 1)
        ))
    ])


# =========================================================
# FEATURE COMBINATIONS
# =========================================================
feature_configs = {
    "textstat": numeric_pipe(textstat_features),
    "overlap": numeric_pipe(overlap_features),
    "ngrams": text_pipe(),

    "textstat+overlap": FeatureUnion([
        ("textstat", numeric_pipe(textstat_features)),
        ("overlap", numeric_pipe(overlap_features)),
    ]),

    "textstat+ngrams": FeatureUnion([
        ("textstat", numeric_pipe(textstat_features)),
        ("ngrams", text_pipe()),
    ]),

    "overlap+ngrams": FeatureUnion([
        ("overlap", numeric_pipe(overlap_features)),
        ("ngrams", text_pipe()),
    ]),

    "all": FeatureUnion([
        ("textstat", numeric_pipe(textstat_features)),
        ("overlap", numeric_pipe(overlap_features)),
        ("ngrams", text_pipe()),
    ]),
}


# =========================================================
# STRATIFIED DOWNSAMPLING
# =========================================================
def downsample_to_size(df, target_col, target_size, random_state=42):
    if target_size > len(df):
        raise ValueError("target_size cannot be larger than dataset size")

    y = df[target_col].astype(str).str.upper()

    sampled_df, _ = train_test_split(
        df,
        train_size=target_size,
        stratify=y,
        random_state=random_state
    )
    return sampled_df.copy()


# =========================================================
# EXPERIMENT DATASETS
# =========================================================
caes_df = data[data["dataset"] == "caes"].copy()
hablacultura_df = data[data["dataset"] == "hablacultura"].copy()
kwiziq_df = data[data["dataset"] == "kwiziq"].copy()

caes_match_kwiziq = downsample_to_size(
    caes_df,
    target_col=target,
    target_size=len(kwiziq_df),
    random_state=random_state
)

caes_match_hablacultura = downsample_to_size(
    caes_df,
    target_col=target,
    target_size=len(hablacultura_df),
    random_state=random_state
)

experiments = {
    "caes_match_kwiziq": caes_match_kwiziq,
    "caes_match_hablacultura": caes_match_hablacultura,
}


# =========================================================
# INSPECT DISTRIBUTIONS
# =========================================================
for name, df in experiments.items():
    print(f"\n=== {name} ===")
    print(df[target].astype(str).str.upper().value_counts().reindex(cefr_order, fill_value=0))


# =========================================================
# CV LOOP
# =========================================================
rows = []

for exp_name, exp_df in experiments.items():
    print(f"\n{'=' * 80}")
    print(f"EXPERIMENT: {exp_name}")
    print(f"{'=' * 80}")

    X_df = exp_df.copy()
    y_str = exp_df[target].astype(str).str.upper().values
    y = np.array([cefr_to_int[v] for v in y_str])

    for feat_name, feat_pipe in feature_configs.items():
        print(f"\n--- Features: {feat_name} ---")

        model = make_model(feat_pipe)

        qwk_list, acc_list, f1w_list, f1m_list = [], [], [], []

        for fold_id, (tr_idx, te_idx) in enumerate(cv.split(X_df, y), start=1):
            X_tr, X_te = X_df.iloc[tr_idx], X_df.iloc[te_idx]
            y_tr, y_te = y[tr_idx], y[te_idx]

            clf = clone(model)
            clf.fit(X_tr, y_tr)
            y_pred = clf.predict(X_te)

            qwk = cohen_kappa_score(y_te, y_pred, weights="quadratic")
            acc = accuracy_score(y_te, y_pred)
            f1w = f1_score(y_te, y_pred, average="weighted", zero_division=0)
            f1m = f1_score(y_te, y_pred, average="macro", zero_division=0)

            qwk_list.append(qwk)
            acc_list.append(acc)
            f1w_list.append(f1w)
            f1m_list.append(f1m)

            print(
                f"Fold {fold_id}: "
                f"QWK={qwk:.4f}, "
                f"acc={acc:.4f}, "
                f"f1_weighted={f1w:.4f}, "
                f"f1_macro={f1m:.4f}"
            )

        row = {
            "experiment": exp_name,
            "dataset": "caes",
            "train_size": len(exp_df),
            "features": feat_name,
            "qwk_mean": float(np.mean(qwk_list)),
            "qwk_std": float(np.std(qwk_list)),
            "acc_mean": float(np.mean(acc_list)),
            "acc_std": float(np.std(acc_list)),
            "f1_weighted_mean": float(np.mean(f1w_list)),
            "f1_weighted_std": float(np.std(f1w_list)),
            "f1_macro_mean": float(np.mean(f1m_list)),
            "f1_macro_std": float(np.std(f1m_list)),
        }
        rows.append(row)

        print(
            f"Mean: "
            f"QWK={row['qwk_mean']:.4f} ± {row['qwk_std']:.4f} | "
            f"acc={row['acc_mean']:.4f} ± {row['acc_std']:.4f} | "
            f"f1_weighted={row['f1_weighted_mean']:.4f} ± {row['f1_weighted_std']:.4f} | "
            f"f1_macro={row['f1_macro_mean']:.4f} ± {row['f1_macro_std']:.4f}"
        )


# =========================================================
# RESULTS
# =========================================================
results = pd.DataFrame(rows).sort_values(
    by=["experiment", "f1_weighted_mean", "qwk_mean", "acc_mean"],
    ascending=[True, False, False, False]
)

print("\n=== FEATURE COMBINATION RESULTS ===")
print(results)

best_per_experiment = results.groupby("experiment", as_index=False).first()

print("\n=== BEST FEATURE SET PER EXPERIMENT ===")
print(best_per_experiment)

print("\n=== LATEX TABLE ===")
print(
    best_per_experiment[
        [
            "experiment",
            "train_size",
            "features",
            "qwk_mean",
            "qwk_std",
            "acc_mean",
            "f1_weighted_mean",
            "f1_macro_mean",
        ]
    ].to_latex(index=False, float_format="%.4f")
)

results.to_csv("exp3_training_size.csv", index=False)

dataset       cefr
caes          A2      5708
              A1      5553
              B1      4489
              B2      3100
              C1      1786
hablacultura  B2       273
              B1       245
              C1        99
              A2        33
kwiziq        B2        59
              B1        54
              C1        39
              A2        33
              A1        21
Name: count, dtype: int64

=== caes_match_kwiziq ===
cefr
A1    55
A2    57
B1    45
B2    31
C1    18
Name: count, dtype: int64

=== caes_match_hablacultura ===
cefr
A1    175
A2    180
B1    141
B2     98
C1     56
Name: count, dtype: int64

EXPERIMENT: caes_match_kwiziq

--- Features: textstat ---
Fold 1: QWK=0.5833, acc=0.4524, f1_weighted=0.4551, f1_macro=0.4585
Fold 2: QWK=0.6790, acc=0.5610, f1_weighted=0.5219, f1_macro=0.5238
Fold 3: QWK=0.7503, acc=0.4878, f1_weighted=0.4650, f1_macro=0.5295
Fold 4: QWK=0.7481, acc=0.5366, f1_weighted=0.5015, f1_macro=0.5206
Fold 5: QWK=0.7252, acc=0.463

## Training Curve Ngrams and Readability+Overlap

In [4]:
import numpy as np
import pandas as pd

from nltk.corpus import stopwords

from sklearn.base import clone
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import cohen_kappa_score, accuracy_score, f1_score


# =========================================================
# SETTINGS
# =========================================================
target = "cefr"
text_col = "lemmas"
dataset_name = "caes"

spanish_stopwords = stopwords.words("spanish")

textstat_features = [
    "fernandez_huerta", "szigriszt_pazos", "gutierrez_polini", "crawford",
    "syllable_count", "lexicon_count", "sentence_count", "words_per_sentence",
    "avg_syllables_per_word", "difficult_words"
]

overlap_features = ["A1_overlap", "A2_overlap", "B1_overlap", "B2_overlap"]

cefr_order = ["A1", "A2", "B1", "B2", "C1"]
cefr_to_int = {c: i for i, c in enumerate(cefr_order)}

n_splits = 5
cv_random_state = 42
sample_random_state = 42

cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=cv_random_state)


# =========================================================
# FINAL MODEL
# =========================================================
def make_model(feature_pipeline):
    return Pipeline([
        ("features", feature_pipeline),
        ("scaler", StandardScaler(with_mean=False)),
        ("clf", LogisticRegression(
            solver="lbfgs",
            penalty="l2",
            C=1.0,
            max_iter=3000,
            class_weight="balanced"
        ))
    ])


# =========================================================
# FEATURE PIPELINES
# =========================================================
def numeric_pipe(cols):
    return Pipeline([
        ("select", FunctionTransformer(lambda X: X[cols], validate=False))
    ])


def text_pipe():
    return Pipeline([
        ("select", FunctionTransformer(lambda X: X[text_col], validate=False)),
        ("tfidf", TfidfVectorizer(
            stop_words=spanish_stopwords,
            ngram_range=(1, 1)
        ))
    ])


# =========================================================
# FEATURE COMBINATIONS
# only the two needed for the training-curve experiment
# =========================================================
feature_configs = {
    "ngrams": text_pipe(),

    "textstat+overlap": FeatureUnion([
        ("textstat", numeric_pipe(textstat_features)),
        ("overlap", numeric_pipe(overlap_features)),
    ]),
}


# =========================================================
# HELPERS
# =========================================================
def make_doubling_sizes(n_total, start_size=20):
    """
    Create sizes: 20, 40, 80, 160, ... up to n_total.
    Always include n_total at the end if not already present.
    """
    sizes = []
    n = start_size

    while n < n_total:
        sizes.append(n)
        n *= 2

    if n_total not in sizes:
        sizes.append(n_total)

    return sizes


def stratified_subsample(df, target_col, n_samples, random_state=42):
    """
    Return a stratified subset of exactly n_samples.
    """
    if n_samples >= len(df):
        return df.sample(frac=1.0, random_state=random_state).copy()

    # train_test_split with train_size=n_samples performs stratified sampling
    idx = np.arange(len(df))
    y = df[target_col].astype(str).str.upper().values

    sub_idx, _ = train_test_split(
        idx,
        train_size=n_samples,
        stratify=y,
        random_state=random_state
    )

    return df.iloc[sub_idx].copy()


def can_run_stratified_kfold(df, target_col, n_splits=5):
    """
    Check whether every class has at least n_splits members.
    """
    counts = df[target_col].astype(str).str.upper().value_counts()
    return counts.min() >= n_splits


# =========================================================
# DATA
# =========================================================
caes_df = data[data["dataset"] == dataset_name].copy()

# optional cleanup
caes_df = caes_df.dropna(subset=[target, text_col]).copy()

# keep only known CEFR labels
caes_df = caes_df[
    caes_df[target].astype(str).str.upper().isin(cefr_order)
].copy()

print(f"Dataset: {dataset_name}")
print(f"Total samples: {len(caes_df)}")
print(caes_df[target].astype(str).str.upper().value_counts().reindex(cefr_order, fill_value=0))

train_sizes = make_doubling_sizes(len(caes_df), start_size=20)
print("\nRequested training sizes:", train_sizes)


# =========================================================
# TRAINING SIZE EXPERIMENT
# =========================================================
fold_rows = []
summary_rows = []

for train_size in train_sizes:
    print(f"\n{'=' * 80}")
    print(f"TRAIN SIZE: {train_size}")
    print(f"{'=' * 80}")

    # stratified subset
    subset_df = stratified_subsample(
        caes_df,
        target_col=target,
        n_samples=train_size,
        random_state=sample_random_state
    ).reset_index(drop=True)

    class_counts = (
        subset_df[target]
        .astype(str)
        .str.upper()
        .value_counts()
        .reindex(cefr_order, fill_value=0)
    )

    print("Class counts in sampled subset:")
    print(class_counts)

    # skip sizes that cannot support 5-fold stratified CV
    if not can_run_stratified_kfold(subset_df, target_col=target, n_splits=n_splits):
        print(
            f"Skipping train_size={train_size}: "
            f"at least one class has fewer than {n_splits} instances."
        )
        continue

    X_df = subset_df.copy()
    y_str = X_df[target].astype(str).str.upper().values
    y = np.array([cefr_to_int[v] for v in y_str])

    for feat_name, feat_pipe in feature_configs.items():
        print(f"\n--- Features: {feat_name} ---")

        model = make_model(feat_pipe)

        qwk_list, acc_list, f1w_list, f1m_list = [], [], [], []

        for fold_id, (tr_idx, te_idx) in enumerate(cv.split(X_df, y), start=1):
            X_tr, X_te = X_df.iloc[tr_idx], X_df.iloc[te_idx]
            y_tr, y_te = y[tr_idx], y[te_idx]

            clf = clone(model)
            clf.fit(X_tr, y_tr)
            y_pred = clf.predict(X_te)

            qwk = cohen_kappa_score(y_te, y_pred, weights="quadratic")
            acc = accuracy_score(y_te, y_pred)
            f1w = f1_score(y_te, y_pred, average="weighted", zero_division=0)
            f1m = f1_score(y_te, y_pred, average="macro", zero_division=0)

            qwk_list.append(qwk)
            acc_list.append(acc)
            f1w_list.append(f1w)
            f1m_list.append(f1m)

            fold_rows.append({
                "dataset": dataset_name,
                "train_size": train_size,
                "features": feat_name,
                "fold": fold_id,
                "qwk": float(qwk),
                "accuracy": float(acc),
                "f1_weighted": float(f1w),
                "f1_macro": float(f1m),
            })

            print(
                f"Fold {fold_id}: "
                f"QWK={qwk:.4f}, "
                f"acc={acc:.4f}, "
                f"f1_weighted={f1w:.4f}, "
                f"f1_macro={f1m:.4f}"
            )

        summary_row = {
            "dataset": dataset_name,
            "train_size": train_size,
            "features": feat_name,
            "qwk_mean": np.mean(qwk_list),
            "qwk_std": np.std(qwk_list),
            "acc_mean": np.mean(acc_list),
            "acc_std": np.std(acc_list),
            "f1_weighted_mean": np.mean(f1w_list),
            "f1_weighted_std": np.std(f1w_list),
            "f1_macro_mean": np.mean(f1m_list),
            "f1_macro_std": np.std(f1m_list),
        }

        summary_rows.append(summary_row)

        print(
            f"Mean: "
            f"QWK={summary_row['qwk_mean']:.4f} ± {summary_row['qwk_std']:.4f} | "
            f"acc={summary_row['acc_mean']:.4f} ± {summary_row['acc_std']:.4f} | "
            f"f1_weighted={summary_row['f1_weighted_mean']:.4f} ± {summary_row['f1_weighted_std']:.4f} | "
            f"f1_macro={summary_row['f1_macro_mean']:.4f} ± {summary_row['f1_macro_std']:.4f}"
        )


# =========================================================
# RESULTS
# =========================================================
fold_results = pd.DataFrame(fold_rows)
summary_results = pd.DataFrame(summary_rows)

if not summary_results.empty:
    summary_results = summary_results.sort_values(
        by=["train_size", "features"]
    )

print("\n=== SUMMARY RESULTS ===")
print(summary_results)

print("\n=== FOLD-LEVEL RESULTS ===")
print(fold_results.head())

# =========================================================
# SAVE CSV
# =========================================================
summary_results.to_csv("exp4_training_curve_summary.csv", index=False)
fold_results.to_csv("exp4_training_curve_folds.csv", index=False)

print("\nSaved:")
print("- exp4_training_curve_summary.csv")
print("- exp4_training_curve_folds.csv")

Dataset: caes
Total samples: 20636
cefr
A1    5553
A2    5708
B1    4489
B2    3100
C1    1786
Name: count, dtype: int64

Requested training sizes: [20, 40, 80, 160, 320, 640, 1280, 2560, 5120, 10240, 20480, 20636]

TRAIN SIZE: 20
Class counts in sampled subset:
cefr
A1    5
A2    6
B1    4
B2    3
C1    2
Name: count, dtype: int64
Skipping train_size=20: at least one class has fewer than 5 instances.

TRAIN SIZE: 40
Class counts in sampled subset:
cefr
A1    11
A2    11
B1     9
B2     6
C1     3
Name: count, dtype: int64
Skipping train_size=40: at least one class has fewer than 5 instances.

TRAIN SIZE: 80
Class counts in sampled subset:
cefr
A1    22
A2    22
B1    17
B2    12
C1     7
Name: count, dtype: int64

--- Features: ngrams ---
Fold 1: QWK=0.8470, acc=0.7500, f1_weighted=0.7635, f1_macro=0.8033
Fold 2: QWK=0.8870, acc=0.8750, f1_weighted=0.8462, f1_macro=0.7270
Fold 3: QWK=0.5979, acc=0.6875, f1_weighted=0.6458, f1_macro=0.6167
Fold 4: QWK=0.4379, acc=0.5625, f1_weighted=0.